In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm

e:\Tensorflow Kernel\tensorflow_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Image Preprocessing
#Vision Transformers require fixed image size.

IMG_SIZE = 224
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
#Load Dataset
train_dataset = datasets.ImageFolder(
    "path_to/AffectNet/Train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    "path_to/AffectNet/Test",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(train_dataset.classes)

In [ ]:
#Load Pretrained Vision Transformer (ViT Base pretrained on ImageNet)
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=num_classes
)

#Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [5]:
#Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [6]:
EPOCHS = 8

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Training Accuracy: {acc:.2f}%")

Epoch 1/8
Loss: 666.2447
Training Accuracy: 46.94%
Epoch 2/8
Loss: 478.8855
Training Accuracy: 64.40%
Epoch 3/8
Loss: 396.7562
Training Accuracy: 70.56%
Epoch 4/8
Loss: 345.9017
Training Accuracy: 74.75%
Epoch 5/8
Loss: 298.6772
Training Accuracy: 78.79%
Epoch 6/8
Loss: 239.8341
Training Accuracy: 83.28%
Epoch 7/8
Loss: 206.5970
Training Accuracy: 85.55%
Epoch 8/8
Loss: 163.2901
Training Accuracy: 88.56%


In [8]:
torch.save(model.state_dict(), "vit_fer_model.pth")

In [9]:
#Evaluate Model
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)

Test Accuracy: 66.1936905909905
